# Paper Figure 5: ASHD vs AEDP Key Models

This notebook regenerates the draft-style Figure 5 from the current processed 148-household ASHD/AEDP comparison results. It does not rerun PyNNLF experiments.

## 1. Purpose, Inputs, And Outputs

Input: `results/01_ashd_aedp_148hh_comparison/ashd_aedp_148hh_fh8_combined_recap.csv`.

Output: `results/01_ashd_aedp_148hh_comparison/figures/paper_figure_ashd_aedp_key_models_test_nrmse.png`.

The figure compares ASHD and AEDP at the shared 1-day forecast horizon. Rows are the current best model per dataset, naive, and ARIMA. Whiskers use `test_nRMSE_stddev`.

## 2. Setup

In [1]:

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_publication_project(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if candidate.name == "journal_article_1" and (candidate / "results").exists() and (candidate / "notebooks").exists():
            return candidate
        nested = candidate / "publication" / "journal_article_1"
        if (nested / "results").exists() and (nested / "notebooks").exists():
            return nested.resolve()
    raise FileNotFoundError("Could not find publication/journal_article_1 from the current working directory.")


PROJECT_DIR = find_publication_project()
RESULTS_DIR = PROJECT_DIR / "results"
print(f"Publication project: {PROJECT_DIR}")

MODEL_LABELS = {
    "m1_naive_hp1": "naive_hp1",
    "m2_snaive_hp2": "snaive_hp2",
    "m3_ets_hp1": "ets_hp1",
    "m4_arima_hp1": "arima_hp1",
    "m6_lr_hp1": "lr_hp1",
    "m7_ann_hp1": "ann_hp1",
    "m8_dnn_hp1": "dnn_hp1",
    "m9_rt_hp3": "rt_hp3",
    "m10_rf_hp1": "rf_hp1",
    "m13_lstm_hp2": "lstm_hp2",
    "m16_prophet_hp1": "prophet_hp1",
    "m17_xgb_hp1": "xgb_hp1",
}
MODEL_ORDER = list(MODEL_LABELS)
KEY_MODELS = {
    "Naive": "m1_naive_hp1",
    "ARIMA": "m4_arima_hp1",
}

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})


def short_model_name(model_name: str) -> str:
    return MODEL_LABELS.get(str(model_name), str(model_name))


def save_figure(fig, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=300, bbox_inches="tight")
    print(f"Saved: {path}")

SECTION_DIR = RESULTS_DIR / "01_ashd_aedp_148hh_comparison"
FIGURES_DIR = SECTION_DIR / "figures"
INPUT_PATH = SECTION_DIR / "ashd_aedp_148hh_fh8_combined_recap.csv"
OUTPUT_PATH = FIGURES_DIR / "paper_figure_ashd_aedp_key_models_test_nrmse.png"
DATASET_ORDER = ["ASHD_148hh_weather", "AEDP_148hh_weather"]
DATASET_LABELS = {
    "ASHD_148hh_weather": "ASHD 148hh",
    "AEDP_148hh_weather": "AEDP 148hh",
}
COLORS = {
    "ASHD_148hh_weather": "#2F4D67",
    "AEDP_148hh_weather": "#EB932C",
}
print(f"Input: {INPUT_PATH}")
print(f"Output: {OUTPUT_PATH}")


Publication project: <local path redacted>
Input: <local path redacted>
Output: <local path redacted>


## 3. Load And Validate Current Results

In [2]:

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Missing processed result file: {INPUT_PATH}")

recap = pd.read_csv(INPUT_PATH)
required_columns = {
    "dataset_label",
    "forecast_horizon_min",
    "model_name",
    "test_nRMSE",
    "test_nRMSE_stddev",
}
missing_columns = sorted(required_columns - set(recap.columns))
if missing_columns:
    raise ValueError(f"Missing required columns in {INPUT_PATH.name}: {missing_columns}")

recap = recap.loc[recap["dataset_label"].isin(DATASET_ORDER)].copy()
recap["forecast_horizon_min"] = pd.to_numeric(recap["forecast_horizon_min"], errors="coerce")
recap = recap.loc[recap["forecast_horizon_min"].eq(1440)].copy()
for column in ["test_nRMSE", "test_nRMSE_stddev"]:
    recap[column] = pd.to_numeric(recap[column], errors="coerce")

if recap.empty:
    raise ValueError("No ASHD/AEDP 1-day rows found in the combined recap.")
if recap[["test_nRMSE", "test_nRMSE_stddev"]].isna().any().any():
    bad = recap.loc[recap[["test_nRMSE", "test_nRMSE_stddev"]].isna().any(axis=1)]
    raise ValueError("Missing numeric nRMSE values found:\n" + bad[["dataset_label", "model_name", "test_nRMSE", "test_nRMSE_stddev"]].to_string(index=False))

for dataset in DATASET_ORDER:
    dataset_rows = recap.loc[recap["dataset_label"].eq(dataset)]
    missing_models = sorted(set(MODEL_ORDER) - set(dataset_rows["model_name"].astype(str)))
    if missing_models:
        raise ValueError(f"{dataset} is missing expected models: {missing_models}")
    for model in KEY_MODELS.values():
        if model not in set(dataset_rows["model_name"].astype(str)):
            raise ValueError(f"{dataset} is missing key model {model}")

summary = recap.groupby("dataset_label", observed=False).agg(
    rows=("model_name", "count"),
    best_test_nRMSE=("test_nRMSE", "min"),
)
display(summary)


                    rows  best_test_nRMSE
dataset_label                            
AEDP_148hh_weather    12        17.776320
ASHD_148hh_weather    12         6.087214


## 4. Select Current Best, Naive, And ARIMA

In [3]:

best_by_dataset = {}
for dataset in DATASET_ORDER:
    dataset_rows = recap.loc[recap["dataset_label"].eq(dataset)].sort_values(["test_nRMSE", "model_name"])
    best_by_dataset[dataset] = str(dataset_rows.iloc[0]["model_name"])

unique_best_models = sorted(set(best_by_dataset.values()))
if len(unique_best_models) != 1:
    raise ValueError(
        "Figure 5 uses a single y-axis label for the best model, but the current best model differs by dataset: "
        + str({DATASET_LABELS[k]: short_model_name(v) for k, v in best_by_dataset.items()})
    )
best_model = unique_best_models[0]

plot_rows = []
row_specs = [(short_model_name(best_model), best_model), ("naive_hp1", KEY_MODELS["Naive"]), ("arima_hp1", KEY_MODELS["ARIMA"])]
for row_label, model_name in row_specs:
    for dataset in DATASET_ORDER:
        match = recap.loc[recap["dataset_label"].eq(dataset) & recap["model_name"].astype(str).eq(model_name)]
        if match.empty:
            raise ValueError(f"Missing {dataset} / {model_name}")
        item = match.iloc[0]
        plot_rows.append({
            "row_label": row_label,
            "dataset_label": dataset,
            "dataset_display": DATASET_LABELS[dataset],
            "model_name": model_name,
            "model_display": short_model_name(model_name),
            "test_nRMSE": float(item["test_nRMSE"]),
            "test_nRMSE_stddev": float(item["test_nRMSE_stddev"]),
        })

plot_df = pd.DataFrame(plot_rows)
display(plot_df)


   row_label       dataset_label  ... test_nRMSE test_nRMSE_stddev
0    xgb_hp1  ASHD_148hh_weather  ...   6.087214          0.546110
1    xgb_hp1  AEDP_148hh_weather  ...  17.776320          2.414256
2  naive_hp1  ASHD_148hh_weather  ...   9.091399          0.904674
3  naive_hp1  AEDP_148hh_weather  ...  24.297913          3.348745
4  arima_hp1  ASHD_148hh_weather  ...  16.178947          1.980822
5  arima_hp1  AEDP_148hh_weather  ...  37.021131          4.116019

[6 rows x 7 columns]


## 5. Create Figure 5

In [4]:

row_order = list(dict.fromkeys(plot_df["row_label"].tolist()))
y = np.arange(len(row_order))
bar_height = 0.34
fig, ax = plt.subplots(figsize=(7.8, 4.4))

for i, dataset in enumerate(DATASET_ORDER):
    subset = plot_df.loc[plot_df["dataset_label"].eq(dataset)].set_index("row_label").reindex(row_order)
    offset = (i - 0.5) * bar_height
    ax.barh(
        y + offset,
        subset["test_nRMSE"],
        xerr=subset["test_nRMSE_stddev"],
        height=bar_height,
        label=DATASET_LABELS[dataset],
        color=COLORS[dataset],
        alpha=0.92,
        capsize=3,
        error_kw={"elinewidth": 0.9, "alpha": 0.85},
    )

ax.set_yticks(y)
ax.set_yticklabels(row_order)
ax.invert_yaxis()
ax.set_xlabel("Test nRMSE (%)")
ax.set_title("ASHD vs AEDP 148-household datasets: key model performance")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=True)
ax.grid(axis="x", alpha=0.25)
ax.grid(axis="y", visible=False)
right_edge = (plot_df["test_nRMSE"] + plot_df["test_nRMSE_stddev"]).max()
ax.set_xlim(0, right_edge * 1.08)
fig.tight_layout()
save_figure(fig, OUTPUT_PATH)
plt.show()


Saved: <local path redacted>
publication\journal_article_1\notebooks\01_ashd_aedp_148hh_comparison\3_create_paper_figure_05_ashd_aedp_key_models.ipynb:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  },


## 6. Saved File Summary

In [5]:

if not OUTPUT_PATH.exists():
    raise FileNotFoundError(f"Expected figure was not created: {OUTPUT_PATH}")
print(f"Figure 5 PNG: {OUTPUT_PATH}")
print(f"Size: {OUTPUT_PATH.stat().st_size:,} bytes")


Figure 5 PNG: <local path redacted>
Size: 75,886 bytes
